In [6]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T

# =====================================
# Random seed to ensure reproducibility
# =====================================

def set_seed(seed=42):
    # 1. Native python seed:
    random.seed(seed)

    # 2. Evironment python seed:
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 3. Numpy seed:
    np.random.seed(seed)
    
    # 4. PyTorch seed (CPU)
    torch.manual_seed(seed)
    
    # 5. PyTorch seed (GPU / CUDA)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # Si usas múltiples GPUs
    
    # 6. CuDNN deterministic for stability in math operations:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42) # Call seed function

# Version of PyTorch:
print(f"PyTorch version: {torch.__version__}")

# Detecting device:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch version: 2.10.0+cu130
Using device: cuda


In [2]:
# ===================================== 
# Importing the CIFAR-10 dataset:
# =====================================

# Create transformations for the dataset:
transforms = T.Compose([
    T.ToTensor(), # Convert images to PyTorch tensors
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalize pixel values
])

# Load the training and test datasets:
train_dataset = torchvision.datasets.CIFAR10(root='../data/raw', train=True, download=True, transform=transforms)    
test_dataset = torchvision.datasets.CIFAR10(root='../data/raw', train=False, download=True, transform=transforms)

# Create the data loaders:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [06:09<00:00, 462kB/s]    


In [ ]:
# ===================================== 
# Defining the Encoder architecture:
# =====================================

class Encoder(nn.Module):
    def __init__(self, latent_dim = 2):
        super().__init__()
        self.latent_dim = latent_dim

        # Layer 1:
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1) # Output: 32 x 16 x 16
        self.batch1 = nn.BatchNorm2d(32) # Batch normalization for faster convergence and stability 
        # Layer 2:
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1) # Output: 64 x 8 x 8
        self.batch2 = nn.BatchNorm2d(64)
        # Layer 3:
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1) # Output: 128 x 4 x 4
        self.batch3 = nn.BatchNorm2d(128)
        # Layer 4:
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1) # Output: 256 x 2 x 2
        self.batch4 = nn.BatchNorm2d(256)

        # Dense fully connected layer:
        self.linear1 = nn.Linear(256*2*2, 1024)
        self.linear2 = nn.Linear(1024, latent_dim)


    def forward(self, x):
        x = F.relu(self.batch1(self.conv1(x)))
        x = F.relu(self.batch2(self.conv2(x)))
        x = F.relu(self.batch3(self.conv3(x)))
        x = F.relu(self.batch4(self.conv4(x)))

        # Flatten the output of the convolutional layers:
        x = x.view(x.size(0), -1)
        x = F.relu(self.linear1(x))
        x = self.linear2(x) # Output: latent_dim (2 in this case)

        return x


In [ ]:
# ===================================== 
# Defining the Decoder architecture:
# =====================================

class Decoder(nn.Module):
    def __init__(self, latent_dims = 2):
        super().__init__()
        self.latent_dims = latent_dims

        # Unflatten the input:
        self.linear1 = nn.Linear(latent_dims, 1024)
        self.linear2 = nn.Linear(1024, 256*2*2)
        self.unflatten = torch.nn.Unflatten(dim=1, unflattened_size=(256, 2, 2)) # Output: 256 x 2 x 2

        # Layer 1:
        self.convT1 = nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1) # Output: 128 x 4 x 4
        self.batchT1 = nn.BatchNorm2d(128)
        # Layer 2:
        self.convT2 = nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1) # Output: 64 x 8 x 8
        self.batchT2 = nn.BatchNorm2d(64)
        # Layer 3:
        self.convT3 = nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1) # Output: 32 x 16 x 16
        self.batchT3 = nn.BatchNorm2d(32)
        # Layer 4:
        self.convT4 = nn.ConvTranspose2d(32, 3, kernel_size=3, stride=2, padding=1, output_padding=1) # Output: 3 x 32 x 32


    def forward(self, z):
        # Unflatten:
        z = F.relu(self.linear1(z))
        z = F.relu(self.linear2(z))
        z = F.relu(self.unflatten(z))

        # Transposed convolutions:
        z = F.relu(self.batchT1(self.convT1(z)))
        z = F.relu(self.batchT2(self.convT2(z)))
        z = F.relu(self.batchT3(self.convT3(z)))

        # Output:
        z = F.tanh(self.convT4(z)) # Tanh for the standardize pixel intensity of [-1 1]

        return z

In [5]:
# ===================================== 
# Defining the Autoencoder wrapper:
# =====================================

class AE(nn.Module):
    def __init__(self, latent_dims):
        super().__init__()
        self.encoder = Encoder(latent_dims) # Initialize the encoder
        self.decoder = Decoder(latent_dims) # Initialize the decoder

    def forward(self, x):
        z = self.encoder(x)
        x_rec = self.decoder(z)

        return x_rec 

In [ ]:
# ===================================== 
# Defining the Training Loop:
# =====================================

def train_epoch_ae(epoch, model, dataloader, optimizer, criterion, device):
    model.train() # Set model to training mode
    epoch_loss = 0

    for imgs, _ in dataloader:
        imgs = imgs.to(device) # Move the images to de GPU or CPU
        recon_imgs = model(imgs) # Reconstructed images

        recon_loss = criterion(recon_imgs, imgs)

        optimizer.zero_grad() # Clean gradients
        recon_loss.backward() # Back propagation
        optimizer.step()      # Update gradients

        epoch_loss += recon_loss.item() # Accumulate loss
        avg_loss = epoch_loss / len(dataloader) # Average loss

    print(f"Epoch: {epoch} | Recon Loss: {avg_loss:.4f}")

    return avg_loss

In [ ]:
# =========================================================== 
# Defining a function to reconstruct images from latent space
# ===========================================================

def recon_images(model, latent_dims, device):
    with torch.no_grad():
        noise = torch.randn(10, latent_dims).to(device)
        imgs = model.decoder(noise).cpu()
        imgs = torchvision.utils.make_grid(imgs, nrow=5).numpy()

        fig, ax = plt.subplots(figsize=(10, 5), dpi=100)
        plt.imshow(np.transpose(imgs, (1, 2, 0)))
        plt.axis('off')
        plt.show()

        